# 🤖 Lesson 4.4 — Transformers and the Attention Mechanism

**AI Crash Course for Absolute Beginners**

Transformers power GPT, BERT, and every modern LLM. In this notebook:
- Implement scaled dot-product attention from scratch
- Understand Query, Key, Value (Q, K, V)
- See how attention creates context-aware representations
- Use Hugging Face Transformers to run a real BERT model
- Visualise attention weights

> Run each cell with **Shift + Enter**.

## ⚠️ Internet Required + Large Download (~440 MB)

This notebook downloads **BERT** and **DistilBERT** pretrained models from Hugging Face.

**Before running:**
- Make sure Colab has internet access (it does by default)
- Parts 4 and 5 each download a model — allow 1–3 minutes per download
- A GPU is **not required** for this notebook (attention math runs fine on CPU)

**If a cell times out or shows a download error**, run it again — Colab caches models after the first download.

In [ ]:
!pip install torch transformers matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("Ready!")

---
## Part 1 — Scaled Dot-Product Attention from Scratch

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V

    Q: queries  (seq_len x d_k)
    K: keys     (seq_len x d_k)
    V: values   (seq_len x d_v)
    """
    d_k = Q.shape[-1]

    # Step 1: compute similarity scores
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_k)  # (seq x seq)

    # Step 2: optional mask (for causal/padding)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: softmax to get attention weights
    weights = F.softmax(scores, dim=-1)

    # Step 4: weighted sum of values
    output = weights @ V

    return output, weights

print("Attention function defined.")
print("Formula: softmax(Q @ K^T / sqrt(d_k)) @ V")

In [ ]:
# Intuition: 4 tokens in a sentence, d_k = 3
# Imagine tokens: ["The", "cat", "sat", "here"]
seq_len = 4
d_k = 3

torch.manual_seed(0)
Q = torch.randn(seq_len, d_k)   # what each token is looking for
K = torch.randn(seq_len, d_k)   # what each token offers as a key
V = torch.randn(seq_len, d_k)   # what each token offers as a value

output, weights = scaled_dot_product_attention(Q, K, V)

print("Attention weights (each row = how much each token attends to others):")
print(weights.detach().numpy().round(3))
print("\n(Each row sums to 1.0)")

# Visualise
tokens = ["The", "cat", "sat", "here"]
plt.figure(figsize=(5, 4))
plt.imshow(weights.detach().numpy(), cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Attention weight")
plt.xticks(range(4), tokens)
plt.yticks(range(4), tokens)
plt.xlabel("Attended to (Keys)")
plt.ylabel("Query token")
plt.title("Attention Map")
plt.tight_layout()
plt.show()

---
## Part 2 — Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.n_heads, self.d_k)
        return x.transpose(1, 2)   # (batch, heads, seq, d_k)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        Q = self.split_heads(self.W_q(x), batch_size)
        K = self.split_heads(self.W_k(x), batch_size)
        V = self.split_heads(self.W_v(x), batch_size)

        attn_output, weights = scaled_dot_product_attention(Q, K, V)

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(batch_size, seq_len, d_model)

        return self.W_o(attn_output), weights


mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(2, 10, 64)   # batch=2, seq=10, d_model=64
out, attn = mha(x)
print(f"Input shape  : {x.shape}")
print(f"Output shape : {out.shape}")
print(f"Attention shape: {attn.shape}  (batch x heads x seq x seq)")

---
## Part 3 — Positional Encoding

In [ ]:
def positional_encoding(seq_len, d_model):
    """Sinusoidal positional encoding (Vaswani et al. 2017)."""
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return PE

PE = positional_encoding(seq_len=50, d_model=128)

plt.figure(figsize=(10, 4))
plt.imshow(PE, aspect="auto", cmap="RdBu")
plt.colorbar()
plt.xlabel("Embedding dimension")
plt.ylabel("Sequence position")
plt.title("Positional Encoding — each position gets a unique fingerprint")
plt.tight_layout()
plt.show()

---
## Part 4 — Real BERT with Hugging Face

In [ ]:
from transformers import pipeline

# Sentiment analysis with a pretrained DistilBERT
sentiment = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

sentences = [
    "I absolutely love this AI course!",
    "This concept is really hard to understand.",
    "The transformer architecture is surprisingly elegant.",
    "I can't believe how bad traffic was today."
]

print("Sentiment Analysis:")
for s in sentences:
    result = sentiment(s)[0]
    emoji  = "😊" if result["label"] == "POSITIVE" else "😞"
    print(f"  {emoji} [{result['label']:8s} {result['score']:.2f}]  '{s}'")

In [ ]:
# Named Entity Recognition
ner = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english",
               aggregation_strategy="simple")

text = "Elon Musk founded SpaceX in California and Tesla in San Carlos."
entities = ner(text)

print(f"Text: {text}\n")
print(f"{'Entity':<20}  {'Type':<10}  Score")
print("-" * 45)
for e in entities:
    print(f"{e['word']:<20}  {e['entity_group']:<10}  {e['score']:.3f}")

In [ ]:
# Masked language model — BERT predicts missing words
fill = pipeline("fill-mask", model="bert-base-uncased")

prompts = [
    "Artificial intelligence will [MASK] the world.",
    "The [MASK] is the powerhouse of the cell."
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    for result in fill(prompt)[:3]:
        print(f"  '{result['token_str']}'  ({result['score']:.3f})")

---
## Part 5 — Visualise Attention Weights on a Real Sentence

In [ ]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert      = BertModel.from_pretrained("bert-base-uncased", output_attentions=True)

text   = "The cat sat on the mat because it was comfortable."
inputs = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = bert(**inputs)

# Layer 11, head 3 (known to capture co-reference)
attn = outputs.attentions[-1][0, 3].numpy()  # (seq x seq)

plt.figure(figsize=(9, 7))
plt.imshow(attn, cmap="Blues")
plt.colorbar(label="Attention weight")
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right", fontsize=8)
plt.yticks(range(len(tokens)), tokens, fontsize=8)
plt.title(f"BERT Layer 12 Head 4 Attention\n'{text}'")
plt.tight_layout()
plt.show()
print("Notice how 'it' attends strongly to 'cat' — that's co-reference resolution!")

---
## Transformer Architecture at a Glance

```
Input tokens
    |
Token Embeddings + Positional Encoding
    |
[Encoder Block] x N
    Multi-Head Self-Attention
    Add & Norm
    Feed-Forward Network
    Add & Norm
    |
Output representations (used for classification, generation, etc.)
```

---
## Challenge Exercises

1. **Question answering**: Try `pipeline("question-answering")` with a context paragraph and ask it a question.
2. **Zero-shot classification**: `pipeline("zero-shot-classification")` — no fine-tuning needed.
3. **Change heads**: In the attention visualiser, change `[0, 3]` to different head indices. What different relationships do other heads capture?

---
**Next lesson:** [Lesson 4.5 — Transfer Learning](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.5-transfer-learning.ipynb)